In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# ✅ Load environment variables from .env file FIRST
load_dotenv()

# Get API key and set it for both variable names (some libraries use different names)
gemini_key = os.getenv("GEMINI_API_KEY")
if gemini_key:
    # Set both GOOGLE_API_KEY and GEMINI_API_KEY for compatibility
    os.environ["GOOGLE_API_KEY"] = gemini_key
    os.environ["GEMINI_API_KEY"] = gemini_key
else:
    raise ValueError("GEMINI_API_KEY not found in environment. Check your .env file.")

# Get database connection parameters
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME")

# Initialize the model
model = init_chat_model("google_genai:gemini-3-flash-preview")

In [6]:
from sqlalchemy import create_engine
from langchain_community.utilities import SQLDatabase
import os

# Get connection parameters
DB_USER = os.getenv("DB_USER", "city_growth_postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "CityGrowthDiagnostics2026")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "postgres")

# ✅ Create PostgreSQL engine directly (no requests.get needed!)
db_uri = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(db_uri)

# Create SQLDatabase wrapper
db = SQLDatabase(engine)

# Test connection
print(f"Connected! Tables: {db.get_usable_table_names()}")

Connected! Tables: ['msa_wages_employment_data']


In [7]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=db, llm=model)

tools = toolkit.get_tools()

for tool in tools:
    print(f"{tool.name}: {tool.description}\n")

sql_db_query: Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.

sql_db_schema: Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3

sql_db_list_tables: Input is an empty string, output is a comma-separated list of tables in the database.

sql_db_query_checker: Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!



In [8]:
system_prompt = """
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run,
then look at the results of the query and return the answer. Unless the user
specifies a specific number of examples they wish to obtain, always limit your
query to at most {top_k} results.

You can order the results by a relevant column to return the most interesting
examples in the database. Never query for all the columns from a specific table,
only ask for the relevant columns given the question.

You MUST double check your query before executing it. If you get an error while
executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the
database.

To start you should ALWAYS look at the tables in the database to see what you
can query. Do NOT skip this step.

Then you should query the schema of the most relevant tables.
""".format(
    dialect=db.dialect,
    top_k=5,
)

In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware 
from langgraph.checkpoint.memory import InMemorySaver 

agent = create_agent(
    model,
    tools,
    system_prompt=system_prompt,
    middleware=[ 
        HumanInTheLoopMiddleware( 
            interrupt_on={"sql_db_query": True}, 
            description_prefix="Tool execution pending approval", 
        ), 
    ], 
    checkpointer=InMemorySaver(), 
)

In [14]:
question = "Show wage trends for Austin from 2010 to 2020"
config = {"configurable": {"thread_id": "1"}} 

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    config, 
    stream_mode="values",
):
    if "__interrupt__" in step: 
        print("INTERRUPTED:") 
        interrupt = step["__interrupt__"][0] 
        for request in interrupt.value["action_requests"]: 
            print(request["description"]) 
    elif "messages" in step:
        step["messages"][-1].pretty_print()
    else:
        pass

================================ Human Message =================================

Show wage trends for Austin from 2010 to 2020
================================== Ai Message ==================================

[]
Tool Calls:
  sql_db_query_checker (f0e332e9-1658-4d35-94bb-00d74f9f9e21)
 Call ID: f0e332e9-1658-4d35-94bb-00d74f9f9e21
  Args:
    query: SELECT year, avg_annual_pay, annual_avg_wkly_wage, area_title FROM msa_wages_employment_data WHERE area_title LIKE '%Austin%' AND year BETWEEN 2010 AND 2020 ORDER BY year;
================================= Tool Message =================================
Name: sql_db_query_checker

SELECT year, avg_annual_pay, annual_avg_wkly_wage, area_title FROM msa_wages_employment_data WHERE area_title LIKE '%Austin%' AND year BETWEEN 2010 AND 2020 ORDER BY year;
================================== Ai Message ==================================

[]
Tool Calls:
  sql_db_query (1096d326-d87f-4c8c-a7fd-c2b6b50c6c0f)
 Call ID: 1096d326-d87f-4c8c-a7fd-c2b6b50c6

In [15]:
from langgraph.types import Command 

for step in agent.stream(
    Command(resume={"decisions": [{"type": "approve"}]}), 
    config,
    stream_mode="values",
):
    if "messages" in step:
        step["messages"][-1].pretty_print()
    elif "__interrupt__" in step:
        print("INTERRUPTED:")
        interrupt = step["__interrupt__"][0]
        for request in interrupt.value["action_requests"]:
            print(request["description"])
    else:
        pass

================================== Ai Message ==================================

[]
Tool Calls:
  sql_db_query (1096d326-d87f-4c8c-a7fd-c2b6b50c6c0f)
 Call ID: 1096d326-d87f-4c8c-a7fd-c2b6b50c6c0f
  Args:
    query: SELECT year, avg_annual_pay, annual_avg_wkly_wage, area_title FROM msa_wages_employment_data WHERE area_title LIKE '%Austin%' AND year BETWEEN 2010 AND 2020 ORDER BY year;
================================== Ai Message ==================================

[]
Tool Calls:
  sql_db_query (1096d326-d87f-4c8c-a7fd-c2b6b50c6c0f)
 Call ID: 1096d326-d87f-4c8c-a7fd-c2b6b50c6c0f
  Args:
    query: SELECT year, avg_annual_pay, annual_avg_wkly_wage, area_title FROM msa_wages_employment_data WHERE area_title LIKE '%Austin%' AND year BETWEEN 2010 AND 2020 ORDER BY year;
================================= Tool Message =================================
Name: sql_db_query

[(2010, Decimal('48706.00'), Decimal('937.00'), 'Austin-Round Rock, TX'), (2010, Decimal('48706.00'), Decimal('937.00'), 